# 3.2 Clustering (Descriptive Analytics)

We apply two clustering algorithms to the prepared Spotify dataset:
- **K-Means** — centroid-based, requires specifying k
- **DBSCAN** — density-based, finds clusters of arbitrary shape and labels noise

Genre columns are dropped before clustering (unsupervised). We then compare discovered clusters against the known genre labels to evaluate structure.

In [1]:
import pandas as pd

In [2]:
# Load the prepared dataset
df = pd.read_csv("df_transformed.csv")
print(f"Shape: {df.shape}")
print(df.dtypes)

Shape: (1960, 19)
valence                  float64
popularity               float64
acousticness             float64
loudness                 float64
duration_ms              float64
danceability             float64
energy                   float64
speechiness              float64
instrumentalness         float64
liveness                 float64
tempo                    float64
explicit                    bool
key                        int64
mode                       int64
time_signature             int64
track_genre_indie-pop       bool
track_genre_pop             bool
track_genre_r-n-b           bool
track_genre_synth-pop       bool
dtype: object


In [3]:
# Reconstruct original genre labels from one-hot encoded columns
genre_cols = [c for c in df.columns if c.startswith('track_genre_')]

# Find the genre of each track
def decode_genre(row):
    for col in genre_cols:
        if row[col] == 1:
            return col.replace('track_genre_', '')
    return 'hip-hop'  # reference category (dropped by drop_first=True)

# Save the related genre of each track 
genre_labels = df.apply(decode_genre, axis=1)
print("Genre distribution:")
print(genre_labels.value_counts())

Genre distribution:
indie-pop    494
pop          488
synth-pop    393
r-n-b        293
hip-hop      292
Name: count, dtype: int64


In [4]:
# Drop all genre columns for clustering
X = df.drop(columns=genre_cols)
print(f"\nFeature matrix shape: {X.shape}")
print("Columns used:", list(X.columns))


Feature matrix shape: (1960, 15)
Columns used: ['valence', 'popularity', 'acousticness', 'loudness', 'duration_ms', 'danceability', 'energy', 'speechiness', 'instrumentalness', 'liveness', 'tempo', 'explicit', 'key', 'mode', 'time_signature']
